# Track N — <topic> (<owner>)

Copy this notebook to `trackN_<topic>.ipynb`. Fill in the hypothesis **before** looking at results, so the write-up stays honest.

**Hypothesis:** what you expect to improve, for which stocks or days, and why. Example: *"The correction should be smaller on month-end days because heavy trading there is expected; a calendar-adjusted volume surprise should fix the month-end weakness."*

**What would count as success:** usually the gate (development log RMSE at least 0.002 better than the incumbent, no worse WAPE, interval below zero). Improving a weak segment without hurting the total is also worth reporting.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, str(Path.cwd()))
from intraday_harness import Harness, leaderboard
from notebooks.modeling_utils import fit_ridge, predict_ridge

NAME = "trackN_my_model"          # unique, short, no spaces
OWNER = "Your name"
DESCRIPTION = "One sentence: what this tries and why"
QUICK = True                      # 2024 only while iterating; set False for the full 2022-2025 run

## Optional: extra features
Anything fitted here must use rows before `fold_year` only. For features across all stocks on a date, use `h.full_panel(...)`.

In [ ]:
h = Harness(quick=QUICK)

def my_features(frame, fold_year, role):
    # Example: return pd.DataFrame({"volume_surprise_sq": frame.cp_volume_surprise20 ** 2}, index=frame.index)
    return pd.DataFrame(index=frame.index)

h.add_features(my_features)
EXTRA_CONTINUOUS = []             # names of the columns you added

## Fit and submit
Replace the ridge below with your model. Tune on `fold.tune` and `fold.valid`, then refit on `fold.refit`. Never fit on `fold.test`.

In [ ]:
for fold in h.folds():
    # --- tuning: choose settings on the validation year ---
    prep = fold.preprocessor(fold.tune, extra_continuous=EXTRA_CONTINUOUS)
    x_tune, x_valid = prep.transform(fold.tune), prep.transform(fold.valid)
    best = None
    for strength in (1e-5, 1e-3, 1e-1):
        beta = fit_ridge(x_tune, fold.target(fold.tune), strength)
        forecast = fold.to_shares(predict_ridge(x_valid, beta), fold.valid, fold.bounds(fold.tune))
        rmse = np.sqrt(np.mean((np.log1p(forecast) - fold.valid.log1p_auction_volume) ** 2))
        best = min(best or (rmse, strength), (rmse, strength))
    # --- final fit through the year before the test year ---
    prep = fold.preprocessor(fold.refit, extra_continuous=EXTRA_CONTINUOUS)
    beta = fit_ridge(prep.transform(fold.refit), fold.target(fold.refit), best[1])
    h.submit(fold, predict_ridge(prep.transform(fold.test), beta), name=NAME, owner=OWNER, description=DESCRIPTION)

## Results against the incumbent

In [ ]:
result = h.report(NAME)           # quick runs are not written to the leaderboard
display(result["summary"])
display(result["intervals"])
display(pd.Series(result["gate"], name="gate", dtype=object))
display(result["segments"])

In [ ]:
leaderboard()

## Findings
Two or three sentences: what happened, where it helped or hurt (segments), and whether it is worth combining with another track.